# SQL 版本：核心分析用标准 SQL 重写

前面的分析全部用 pandas 完成。这一份把其中 **“取数 + 聚合”** &#8203;这部分用**标准 SQL** 重新实现一遍。

**两条说明**

1. **不替换 Python。**&#8203; 前面的 notebook 一行都不动，这里只是同一批业务问题的另一种实现。
2. **不重复维护 SQL。**&#8203; SQL 逻辑全部写在 `sql/analysis.sql` 里，本 notebook 只负责**建库 → 执行 → 与 Python 结果对账**，不复制任何 SQL 代码。


## 1. 建库：把清洗后的数据导入 SQLite

使用 Python **内置**的 SQLite 轻量数据库，把 `cleaned_data.csv` 导入后，就可以用标准 SQL 查询了。


In [23]:
# 建库：把 cleaned_data.csv 导入 SQLite
import sqlite3
import time
from pathlib import Path
import pandas as pd

# 定位项目根目录
def find_project_root():
    if '__vsc_ipynb_file__' in globals():
        start = Path(globals()['__vsc_ipynb_file__']).parent
    else:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('找不到项目根目录（需同时含 data/ 和 src/）')

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned_data.csv'
DB_PATH   = PROJECT_ROOT / 'data' / 'ecommerce.db'

In [24]:
# 1. 读入清洗后的数据
t0 = time.time()
df = pd.read_csv(DATA_PATH)
print(f"📊 读入 {len(df):,} 行 × {df.shape[1]} 列")

# 2. 布尔转 0/1，方便 SQL 里直接比较
df['is_suspicious_bot'] = df['is_suspicious_bot'].astype(int)

# 3. 写入 SQLite
with sqlite3.connect(DB_PATH) as conn:
    df.to_sql('user_behavior', conn, index=False, if_exists='replace', chunksize=10_000)
print(f"✅ 表 user_behavior 写入完成")

#  4. 建索引：窗口函数 PARTITION BY user_id ORDER BY datetime 靠它提速
with sqlite3.connect(DB_PATH) as conn:
    conn.execute('CREATE INDEX IF NOT EXISTS idx_user_time ON user_behavior(user_id, datetime)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_behavior  ON user_behavior(behavior_type)')
    conn.commit()
print(f"✅ 索引建立完成（总耗时 {time.time()-t0:.1f}s）")

📊 读入 999,530 行 × 10 列
✅ 表 user_behavior 写入完成
✅ 索引建立完成（总耗时 1.3s）


### 1.1 建库验收

建库完成后核对：行数、用户数、时间范围和行为分布


In [25]:
#  验收：与 Python 版基线逐项核对
with sqlite3.connect(DB_PATH) as conn:
    summary = pd.read_sql('''
        SELECT COUNT(*)                    AS 总行数,
               COUNT(DISTINCT user_id)     AS 唯一用户数,
               COUNT(DISTINCT item_id)     AS 唯一商品数,
               COUNT(DISTINCT category_id) AS 唯一品类数,
               MIN(datetime)               AS 最早时间,
               MAX(datetime)               AS 最晚时间
        FROM user_behavior
    ''', conn)

    dist = pd.read_sql('''
        SELECT behavior_type AS 行为类型,
               COUNT(*)      AS 条数
        FROM user_behavior
        GROUP BY behavior_type
        ORDER BY 条数 DESC
    ''', conn)

summary


,总行数,唯一用户数,唯一商品数,唯一品类数,最早时间,最晚时间
0,999530,9739,398971,5793,2017-11-25 00:00:00,2017-12-03 23:59:50


In [26]:
dist

,行为类型,条数
0,pv,895636
1,cart,55447
2,fav,28088
3,buy,20359


## 2. 执行 SQL：读取 `sql/analysis.sql`

SQL 逻辑全部写在 `sql/analysis.sql` 里，本 notebook **不复制** SQL 代码——只在下面把该文件读进来，按 `-- @query: 名称` 标记切成多段，逐段执行并与 Python 结果对账。

In [27]:
# 读取 sql/analysis.sql，按 -- @query: 名称 切成多段
import re

SQL_PATH = PROJECT_ROOT / 'sql' / 'analysis.sql'

def load_sql_queries(path=SQL_PATH):
    """把 .sql 文件按 `-- @query: <name>` 标记切成 {name: sql} 字典。"""
    text = path.read_text(encoding='utf-8')
    parts = re.split(r'^--\s*@query:\s*(\w+)\s*$', text, flags=re.M)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

QUERIES = load_sql_queries()
print(f"✅ 从 {SQL_PATH.relative_to(PROJECT_ROOT)} 载入 {len(QUERIES)} 段查询：{list(QUERIES)}")

def run_sql(name):
    """执行指定名称的 SQL 段，返回 DataFrame。"""
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql(QUERIES[name], conn)


✅ 从 sql/analysis.sql 载入 4 段查询：['01_funnel_penetration', '02_rf_segments', '03_zero_gap_rate', '04_observation_features']


### 2.1 漏斗：并行渗透率（对应 03 notebook §5.2）

**为什么是「并行」而不是「逐级」**

标准的漏斗是「浏览 → 加购 → 购买」逐级收口。但这个平台的实际情况是：**83% 的购买根本不经过加购**——加购和收藏都不是必经环节。

所以这里不用逐级漏斗口径，改用**并行渗透率**：四类行为各算各的去重用户数，分母统一是全平台 9,739 人。四个数加起来会超过 100%，因为同一个用户可能既浏览、又加购、又购买。

**对账方式**：SQL 结果与 `src/` 里的 Python 实现（`compute_behavior_penetration`）逐项比对——用户数和渗透率都必须完全一致。


In [28]:
# 执行 SQL 第 1 段，并与 Python 实现逐项对账
# 1) SQL 求解
result_sql = run_sql('01_funnel_penetration')

# 2) 用 src/ 里的 Python 实现再算一遍（同一份数据、同一个口径）
from ecommerce_conversion_analysis.user_funnel import compute_behavior_penetration

df_raw = pd.read_csv(DATA_PATH)
penetration_py, total_users_py = compute_behavior_penetration(df_raw)

# 3) 并排对账
compare = (
    result_sql
    .rename(columns={'users': 'SQL用户数', 'penetration_pct': 'SQL渗透率'})
    .merge(
        penetration_py.rename(columns={'行为': 'behavior',
                                       '去重用户数': 'Python用户数',
                                       '渗透率(%)': 'Python渗透率'}),
        on='behavior', how='outer',
    )
)
# merge(how='outer') 会按主键字母序重排，这里显式还原业务顺序
# 用 reindex 而不是 sort：缺失的行会以 NaN 保留，不会静默消失
compare = compare.set_index('behavior').reindex(['pv', 'fav', 'cart', 'buy']).reset_index()

compare['一致'] = (compare['SQL用户数'] == compare['Python用户数']) & \
                  (compare['SQL渗透率'] == compare['Python渗透率'])

print(f"分母核对 · Python 版总活跃用户 = {total_users_py:,} 人")
print('✅ 全部一致' if compare['一致'].all() else '❌ 存在差异，需排查')
compare



分母核对 · Python 版总活跃用户 = 9,739 人
✅ 全部一致


,behavior,SQL用户数,SQL渗透率,Python用户数,Python渗透率,一致
0,pv,9706,99.66,9706,99.66,True
1,fav,3882,39.86,3882,39.86,True
2,cart,7323,75.19,7323,75.19,True
3,buy,6689,68.68,6689,68.68,True


### 2.2 用户分层：买家 R×F 九层（对应 03 notebook §6.4）

标准 RFM 是三个维度，这里**只用 R 和 F 两个**，理由如下：

- **R（最近一次购买距今几天）**&#8203;
- **F（买了多少次）**&#8203;
- **M（消费金额）——剔除**。这份数据**没有价格字段**，M 只能用"买过多少件不同商品"来代理。而这个代理指标和 F 高度共线（Spearman ρ = 0.985，90.3% 的用户两个值完全一样），**留着等于把同一件事算两遍权重**。

两个维度各切三档，交叉出 3×3 = 9 层。切点与 Python 版 `R_BINS` / `F_BINS` 完全一致。


In [29]:
# 执行 SQL 第 2 段，并与 Python 实现逐格对账
result_sql = run_sql('02_rf_segments').rename(columns={
    'r_segment': 'R 段位', 'single_buyers': '单次', 'repeat_buyers': '复购',
    'frequent_buyers': '高频复购', 'total': '合计'})

# Python 版：九层交叉表
from ecommerce_conversion_analysis.rfm import (
    build_buyer_rfm, segment_buyers, compute_segment_grid)

df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])   # build_buyer_rfm 需要时间类型
rfm_py  = segment_buyers(build_buyer_rfm(df_raw))
grid_py = (compute_segment_grid(rfm_py)
           .drop(index='合计')                    # 只去掉合计「行」，保留合计「列」
           .reset_index().rename(columns={'R_seg': 'R 段位'}))

# 并排对账（merge 会重排序，最后统一 reindex 还原）
compare = result_sql.merge(grid_py, on='R 段位', suffixes=('·SQL', '·Python'))
compare['九格一致'] = True
for c in ['单次', '复购', '高频复购', '合计']:
    compare['九格一致'] &= (compare[f'{c}·SQL'] == compare[f'{c}·Python'])
compare = (compare.set_index('R 段位')
                  .reindex(['1 天以内', '1–4 天', '4 天以上'])
                  .reset_index())

print(f"九层加总 · SQL = {result_sql['合计'].sum():,} ｜ Python = {len(rfm_py):,} 人")
print('✅ 九格全部一致' if compare['九格一致'].all() else '❌ 存在差异，需排查')
compare


九层加总 · SQL = 6,689 ｜ Python = 6,689 人
✅ 九格全部一致


,R 段位,单次·SQL,复购·SQL,高频复购·SQL,合计·SQL,单次·Python,复购·Python,高频复购·Python,合计·Python,九格一致
0,1 天以内,323,700,736,1759,323,700,736,1759,True
1,1–4 天,831,1158,843,2832,831,1158,843,2832,True
2,4 天以上,1106,757,235,2098,1106,757,235,2098,True


### 2.3 零间隙率：老客 vs 新客（对应 03 notebook §7.3）

**「零间隙」是什么**：这次购买之前，自上一次购买以来，用户**一次浏览、收藏、加购都没有**——直接就下单了。也就是"不比较，直接买"。

**"逐行为累计 + 跨购买取差"**&#8203;：

1. 先给每行打一个同秒排序权重：`pv=0 < fav=1 < cart=2 < buy=3`——**保证同一秒里购买排在非购买行为之后**，否则计数会错
2. 用 `SUM() OVER (... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` 算累计非购买次数
3. 只看购买事件，用 `LAG()` 取该用户**上一次购买时的累计值**
4. 两次相减 = 这一单之前"逛了多少次"


In [30]:
# 执行 SQL 第 3 段，并与 Python 实现逐项对账
result_sql = run_sql('03_zero_gap_rate')

# Python 版：复现 03 notebook 7.3 的 H2 检验口径
from ecommerce_conversion_analysis.pre_purchase import compute_pre_purchase_gaps

pre  = compute_pre_purchase_gaps(df_raw)
freq = pre.groupby('user_id')['purchase_seq'].max()
first_buyers  = freq[freq == 1].index
repeat_buyers = freq[freq >= 2].index

n_first = len(first_buyers)
x_first = int((pre[pre['user_id'].isin(first_buyers)]['gap_before_purchase'] == 0).sum())

repeat_pre = pre[pre['user_id'].isin(repeat_buyers) & (pre['purchase_seq'] >= 2)]
n_repeat = len(repeat_buyers)
x_repeat = int(repeat_pre[repeat_pre['gap_before_purchase'] == 0]['user_id'].nunique())

# 并排对账
py_result = pd.DataFrame({
    'buyer_type':      ['first_time', 'repeat'],
    '组别':            ['新客（只买 1 次）', '老客（买 ≥2 次）'],
    'Python用户数':    [n_first, n_repeat],
    'Python零间隙数':  [x_first, x_repeat],
    'Python零间隙率':  [round(x_first / n_first * 100, 2),
                        round(x_repeat / n_repeat * 100, 2)],
})

compare = (result_sql
           .rename(columns={'n_users': 'SQL用户数', 'n_zero_gap': 'SQL零间隙数',
                            'zero_gap_pct': 'SQL零间隙率'})
           .merge(py_result, on='buyer_type'))

compare['一致'] = ((compare['SQL用户数']    == compare['Python用户数'])
                & (compare['SQL零间隙数']  == compare['Python零间隙数'])
                & (compare['SQL零间隙率']  == compare['Python零间隙率']))

print(f"购买事件总数 = {len(pre):,} ｜ 买家总数 = {freq.size:,}")
print('✅ 全部一致' if compare['一致'].all() else '❌ 存在差异，需排查')
compare


购买事件总数 = 20,359 ｜ 买家总数 = 6,689
✅ 全部一致


,buyer_type,SQL用户数,SQL零间隙数,SQL零间隙率,组别,Python用户数,Python零间隙数,Python零间隙率,一致
0,first_time,2260,74,3.27,新客（只买 1 次）,2260,74,3.27,True
1,repeat,4429,1771,39.99,老客（买 ≥2 次）,4429,1771,39.99,True


### 2.4 观察窗特征表（对应 04 notebook §1.4）

这段把喂给购买预测模型的 **9 个特征**用 SQL 构造出来。

**最关键的一条红线是时间边界。**&#8203; 观察窗（11/25–12/01，7 天）算特征，预测窗（12/02–12/03，2 天）出标签——特征表里绝不能沾到预测窗的任何信息，否则就是**标签泄漏**，模型会给出虚假的高分。

从「全量活跃用户」出发去 LEFT JOIN 观察窗的聚合结果，不能直接从观察窗数据出发。因为**有用户在观察窗里没留下任何行为**——如果从观察窗出发，这些人就整个丢了，样本数会不对。


In [31]:
#  执行 SQL 第 4 段，并与 Python 特征表逐行逐列对账
X_sql = run_sql('04_observation_features')
print(f"SQL 特征表：{X_sql.shape[0]:,} 行 × {X_sql.shape[1] - 1} 列")

# Python 版：完整复现 04 notebook §1.4 的特征工程
import numpy as np

OBS_START, OBS_END = '2017-11-25 00:00:00', '2017-12-01 23:59:59'
obs = df_raw[(df_raw['datetime'] >= OBS_START) & (df_raw['datetime'] <= OBS_END)]
all_users = set(df_raw['user_id'])
OBS_END_DATE = pd.Timestamp('2017-12-01')

cnt = (obs.groupby(['user_id', 'behavior_type']).size().unstack(fill_value=0)
          .reindex(columns=['pv', 'fav', 'cart', 'buy'], fill_value=0)
          .rename(columns={'pv': 'pv_cnt', 'fav': 'fav_cnt',
                           'cart': 'cart_cnt', 'buy': 'obs_buy_cnt'}))
active_days = (obs.assign(date=obs['datetime'].dt.normalize())
                  .groupby('user_id')['date'].nunique().rename('active_days'))
last_active = obs.groupby('user_id')['datetime'].max().dt.normalize()
days_since  = (OBS_END_DATE - last_active).dt.days.rename('days_since_last_active')
n_cats = (obs.loc[obs['behavior_type'] == 'pv']
             .groupby('user_id')['category_id'].nunique().rename('n_cats'))
cart_depth = (cnt['cart_cnt'] / cnt['pv_cnt'].replace(0, np.nan)).fillna(0).rename('cart_depth')
last_buy = (obs.loc[obs['behavior_type'] == 'buy']
               .groupby('user_id')['datetime'].max().dt.normalize())
obs_recency = (OBS_END_DATE - last_buy).dt.days.rename('obs_recency_days')

X_py = (cnt.join([active_days, days_since, n_cats, cart_depth, obs_recency], how='outer')
           .reindex(sorted(all_users)))
X_py['obs_recency_days'] = X_py['obs_recency_days'].fillna(8)   # 哨兵值：窗口内从没买过
X_py = X_py.fillna(0).reset_index().rename(columns={'index': 'user_id'})
print(f"Python 特征表：{X_py.shape[0]:,} 行 × {X_py.shape[1] - 1} 列，"
      f"缺失值 {int(X_py.isna().sum().sum())}")

# 逐列对账
merged = X_py.merge(X_sql, on='user_id', suffixes=('_py', '_sql'))
int_cols = ['pv_cnt', 'fav_cnt', 'cart_cnt', 'obs_buy_cnt', 'active_days',
            'days_since_last_active', 'n_cats', 'obs_recency_days']
check = pd.DataFrame({
    '特征': int_cols + ['cart_depth'],
    '一致': [(merged[f'{c}_py'] == merged[f'{c}_sql']).all() for c in int_cols]
            + [np.allclose(merged['cart_depth_py'], merged['cart_depth_sql'],
                           rtol=1e-9, atol=1e-12)],
})

# 边界检查：只在预测窗出现、观察窗零行为的用户
only_pred = set(df_raw['user_id']) - set(obs['user_id'])
print(f"边界检查 · 观察窗零行为、仅在预测窗出现的用户：{len(only_pred)} 人")
print('✅ 9 个特征全部一致' if check['一致'].all() else '❌ 存在差异，需排查')
check


SQL 特征表：9,739 行 × 9 列
Python 特征表：9,739 行 × 9 列，缺失值 0
边界检查 · 观察窗零行为、仅在预测窗出现的用户：1 人
✅ 9 个特征全部一致


,特征,一致
0,pv_cnt,True
1,fav_cnt,True
2,cart_cnt,True
3,obs_buy_cnt,True
4,active_days,True
5,days_since_last_active,True
6,n_cats,True
7,obs_recency_days,True
8,cart_depth,True
